# MobileNetV2 — Concrete Crack Classification Benchmark

Transfer-learning benchmark of **MobileNetV2** on the Özgenel *Concrete Crack Images for Classification* dataset (Mendeley, id `5y9wdsg2zt`, v2): 40,000 RGB images, 227×227 px, two balanced classes — **Positive** (crack, 20,000) and **Negative** (no crack, 20,000) — cropped from 458 photographs of METU campus buildings.

This notebook covers the first of the five candidate models and serves as the agreed calibration checkpoint. It fine-tunes MobileNetV2 via transfer learning and reports the full metric set: **accuracy** (with precision, recall, F1 and a confusion matrix), **inference latency**, **model size**, **computational cost**, and **memory usage**. The remaining models (MobileNetV3, EfficientNet-Lite, SqueezeNet, ShuffleNet) will be run through this exact pipeline so that the cross-model comparison is like-for-like.

**Reading guide.** Markdown cells document the methodology and the reasoning behind each measurement decision; code cells run top to bottom. The only long-running cell is training (~2–4 minutes on a CUDA GPU).

In [ ]:
# === 1. Imports, seed, configuration ===================================
import os, time, copy, random, tempfile, gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from thop import profile
import pandas as pd

# --- reproducibility: fix the main sources of randomness (Python, NumPy, PyTorch seeds) ---
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# --- experiment configuration ---
DATA_DIR        = r".\data"
IMG_SIZE        = 224          # MobileNetV2 input size (as trained on ImageNet)
BATCH_SIZE      = 64
EPOCHS          = 3            # the dataset saturates quickly; 2-3 epochs suffice
LR              = 1e-4         # low LR = careful fine-tuning of pretrained features
NUM_WORKERS     = 4            # parallel data-loading processes
FREEZE_BACKBONE = False        # False = fine-tune the entire network
MODEL_NAME      = "MobileNetV2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Device: cuda | NVIDIA GeForce RTX 3070 Ti Laptop GPU


## 2. Data: normalization, augmentation, and the train/val/test split

- **ImageNet normalization.** The network starts from ImageNet-pretrained weights, so inputs must match the distribution those weights were trained on: pixels scaled to [0, 1], then standardized per channel with the ImageNet statistics (mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]`).
- **Two transform pipelines.** The training pipeline adds a single augmentation — random horizontal flip, which is label-preserving for crack imagery. The validation and test pipelines are fully deterministic (resize + normalize only), so every reported metric is measured on unaugmented data.
- **Stratified 70/15/15 split.** Images are split into train/validation/test so that the Positive/Negative ratio is preserved in each subset. A fixed `random_state` makes the partition exactly reproducible.
- **Disclosure — crop-level splitting.** The 40,000 crops derive from 458 source photographs, and the split is performed at the crop level, which is the standard convention for this dataset (the public release provides no crop-to-photograph mapping, so a photograph-grouped split is not feasible). Crops from the same photograph can therefore appear on both sides of the split; the reported accuracy reflects in-distribution performance and may be optimistic for entirely new surfaces. This convention applies equally to published results on this dataset and does not affect the *relative* comparison between the five candidate models, which share the identical split.

In [11]:
# === 2. Data loading and stratified split ==============================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),      # the only augmentation
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Two "views" of the same folder with different transforms.
# ImageFolder sorts classes alphabetically -> Negative=0, Positive=1.
full_train_view = datasets.ImageFolder(DATA_DIR, transform=train_tf)
full_eval_view  = datasets.ImageFolder(DATA_DIR, transform=eval_tf)
print("Classes:", full_train_view.classes, "-> idx", full_train_view.class_to_idx)

targets = np.array(full_train_view.targets)
idx = np.arange(len(targets))

# 70 / 15 / 15, stratified
train_idx, temp_idx = train_test_split(idx, test_size=0.30,
                                        stratify=targets, random_state=SEED)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50,
                                        stratify=targets[temp_idx], random_state=SEED)

train_ds = Subset(full_train_view, train_idx)   # augmented
val_ds   = Subset(full_eval_view,  val_idx)     # deterministic
test_ds  = Subset(full_eval_view,  test_idx)    # deterministic

mk = lambda ds, sh: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               persistent_workers=(NUM_WORKERS > 0))
train_loader, val_loader, test_loader = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

Classes: ['Negative', 'Positive'] -> idx {'Negative': 0, 'Positive': 1}
train 28000 | val 6000 | test 6000


## 3. Model: transfer learning with a replaced classification head

The torchvision implementation of MobileNetV2 consists of a convolutional feature extractor (pretrained on ImageNet) and a classifier whose final linear layer outputs 1,000 ImageNet classes. That layer is replaced with a fresh `Linear(1280, 2)` for the binary task.

With `FREEZE_BACKBONE = False` the entire network is fine-tuned at a low learning rate, which on easy datasets typically yields slightly higher accuracy than training the new head alone. Setting the flag to `True` would freeze the pretrained features and train only the head.

In [12]:
# === 3. Model assembly =================================================
def build_model(freeze_backbone=FREEZE_BACKBONE):
    weights = MobileNet_V2_Weights.IMAGENET1K_V1
    model = mobilenet_v2(weights=weights)
    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
    in_feats = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_feats, 2)     # 1000 -> 2 classes
    return model.to(device)

model = build_model()
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_train:,} / {n_total:,}")

Trainable parameters: 2,226,434 / 2,226,434


## 4. Training

Standard cross-entropy loss with the Adam optimizer. After each epoch the model is evaluated on the validation split, and the weights with the best validation accuracy are kept for all subsequent measurements. The test set plays no role in model selection.

In [13]:
# === 4. Training loop ==================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward(); optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += x.size(0)
    return loss_sum / total, correct / total

best_acc, best_state = 0.0, None
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    print(f"epoch {ep}: train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f}")
    if va_acc > best_acc:
        best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)    # restore the best-validation weights
print("Best validation accuracy:", round(best_acc, 4))

epoch 1: train loss 0.0148 acc 0.9959 | val loss 0.0035 acc 0.9985
epoch 2: train loss 0.0037 acc 0.9988 | val loss 0.0015 acc 0.9992
epoch 3: train loss 0.0016 acc 0.9995 | val loss 0.0020 acc 0.9995
Best validation accuracy: 0.9995


## 5. Classification quality on the held-out test set

Accuracy is reported together with precision, recall and F1 for the **Positive (crack)** class, plus a full confusion matrix. Recall receives separate attention deliberately: in structural inspection a missed crack (false negative) is costlier than a false alarm, so the fraction of real cracks the model detects matters on its own.

**Note on mAP and Top-5 accuracy.** Neither metric applies to binary classification: Top-5 accuracy is trivially 100% when only two classes exist, and mAP is an object-detection metric that requires bounding-box annotations, which this dataset does not contain. Top-1 accuracy is reported instead.

In [14]:
# === 5. Quality metrics on the TEST set ================================
@torch.no_grad()
def collect_preds(loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        out = model(x.to(device))
        ps.append(out.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

pos = full_eval_view.class_to_idx["Positive"]
y_true, y_pred = collect_preds(test_loader)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", pos_label=pos)
cm = confusion_matrix(y_true, y_pred)

print(f"Test accuracy: {acc:.4f}")
print(f"Precision {prec:.4f} | Recall {rec:.4f} | F1 {f1:.4f}")
print("Confusion matrix [rows = true, cols = predicted], class order:",
      full_eval_view.classes)
print(cm)
print(classification_report(y_true, y_pred, target_names=full_eval_view.classes))

Test accuracy: 0.9990
Precision 0.9987 | Recall 0.9993 | F1 0.9990
Confusion matrix [rows = true, cols = predicted], class order: ['Negative', 'Positive']
[[2996    4]
 [   2 2998]]
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00      3000
    Positive       1.00      1.00      1.00      3000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



## 6. Inference latency: measurement protocol

Naive timing of GPU code is misleading; the protocol below avoids the three standard pitfalls:

1. **Warm-up.** The first GPU passes include CUDA context initialization and kernel autotuning. Twenty untimed warm-up runs precede the measurement.
2. **Synchronization.** CUDA calls are asynchronous — control returns to Python before the GPU finishes. `torch.cuda.synchronize()` is called around every timed pass so the timer measures completed computation, not queue submission.
3. **Median over mean.** Background OS activity produces occasional upward outliers. The median of 100 timed runs is reported; p95 is included to make jitter visible.

Latency is measured at **batch size 1** (single image) — the deployment-relevant setting — separately on GPU and CPU. All figures are **eager-mode FP32 PyTorch**; no TensorRT, `torch.compile`, or quantization is applied. Optimized deployment runtimes would be substantially faster; these figures are for like-for-like comparison between the candidate models.

In [15]:
# === 6. Latency measurement (GPU and CPU): warmup + sync + median ======
def measure_latency(model, device_, n_warmup=20, n_runs=100, img_size=IMG_SIZE):
    m = copy.deepcopy(model).to(device_).eval()   # a copy, so the main model stays on its device
    x = torch.randn(1, 3, img_size, img_size, device=device_)
    is_cuda = device_.type == "cuda"
    with torch.no_grad():
        for _ in range(n_warmup):                 # warm-up, untimed
            _ = m(x)
        if is_cuda:
            torch.cuda.synchronize()
        times = []
        for _ in range(n_runs):
            if is_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = m(x)
            if is_cuda:
                torch.cuda.synchronize()          # wait for actual completion
            times.append((time.perf_counter() - t0) * 1000.0)   # -> ms
    t = np.array(times)
    return {"median_ms": float(np.median(t)),
            "p95_ms":    float(np.percentile(t, 95)),
            "mean_ms":   float(t.mean())}

lat_gpu = measure_latency(model, torch.device("cuda")) if torch.cuda.is_available() else None
lat_cpu = measure_latency(model, torch.device("cpu"))
print("GPU latency, batch=1 (ms):", lat_gpu)
print("CPU latency, batch=1 (ms):", lat_cpu)

GPU latency, batch=1 (ms): {'median_ms': 5.335499998182058, 'p95_ms': 5.779674995574169, 'mean_ms': 5.327488000621088}
CPU latency, batch=1 (ms): {'median_ms': 17.36235001590103, 'p95_ms': 22.527359993546266, 'mean_ms': 19.328780999057926}


## 7. Model size, computational cost, and memory usage

- **Model size** — the trained weights (`state_dict`) saved to disk, reported in decimal megabytes (1 MB = 10⁶ bytes).
- **Computational cost** — measured with the `thop` profiler, which counts multiply–accumulate operations (**MACs**). One MAC = one multiplication + one addition, so **FLOPs ≈ 2 × MACs**; both figures are reported and labeled explicitly. The same profiler is used for all five models so the comparison stays consistent.
- **Memory usage** — peak GPU memory allocated during a single-image forward pass.

In [16]:
# === 7. Size / FLOPs (MACs) / peak memory ==============================
# on-disk size of the trained weights
tmp = os.path.join(tempfile.gettempdir(), "mobilenetv2_crack.pt")
torch.save(model.state_dict(), tmp)
size_mb = os.path.getsize(tmp) / 1e6

# MACs and parameter count (thop profiles a CPU copy)
m_cpu = copy.deepcopy(model).cpu().eval()
macs, params = profile(m_cpu, inputs=(torch.randn(1, 3, IMG_SIZE, IMG_SIZE),), verbose=False)
gmacs = macs / 1e9

# peak GPU memory for a single forward pass
peak_mb = None
if torch.cuda.is_available():
    # release training-time allocations (optimizer state, best-checkpoint copy)
    # so the peak reflects model weights + inference activations only
    for _v in ("optimizer", "best_state"):
        if _v in globals():
            del globals()[_v]
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device="cuda"))
    torch.cuda.synchronize()
    peak_mb = torch.cuda.max_memory_allocated() / 1e6

print(f"Model size:          {size_mb:.2f} MB")
print(f"Computational cost:  {gmacs:.3f} GMACs  (~{2*gmacs:.3f} GFLOPs)")
print(f"Parameters:          {params/1e6:.2f} M")
print(f"Peak GPU memory (1 img): {peak_mb:.2f} MB" if peak_mb is not None else "No GPU available")

Model size:          9.15 MB
Computational cost:  0.326 GMACs  (~0.652 GFLOPs)
Parameters:          2.23 M
Peak GPU memory (1 img): 47.33 MB


## 8. Results

The summary below follows the requested metric order — **Accuracy, Inference Latency, Model Size, Computational Cost, Memory Usage** — followed by supplementary quality metrics (precision / recall / F1, parameter count).

A note on interpreting accuracy: reported transfer-learning results on this dataset are typically above 99%, i.e. the dataset is nearly saturated. The accuracy column is therefore expected to be a near-tie across all five candidate models, and the meaningful differentiation between them will come from latency, model size, and computational cost.

In [ ]:
# === 8. Summary table (requested metric order) and CSV export ==========
def fmt_ms(d, key):
    return f"{d[key]:.2f} ms" if d else "n/a"

summary_rows = [
    ("Accuracy (top-1)",                          f"{acc*100:.2f} %"),
    ("Inference Latency - GPU (batch=1, median)", fmt_ms(lat_gpu, "median_ms")),
    ("Inference Latency - GPU (batch=1, p95)",    fmt_ms(lat_gpu, "p95_ms")),
    ("Inference Latency - CPU (batch=1, median)", fmt_ms(lat_cpu, "median_ms")),
    ("Inference Latency - CPU (batch=1, p95)",    fmt_ms(lat_cpu, "p95_ms")),
    ("Model Size (weights on disk)",              f"{size_mb:.2f} MB"),
    ("Computational Cost (GMACs / GFLOPs)",       f"{gmacs:.3f} GMACs  (~{2*gmacs:.3f} GFLOPs)"),
    ("Memory Usage (peak GPU, single image)",     f"{peak_mb:.1f} MB" if peak_mb is not None else "n/a"),
    ("Precision (Positive = crack)",              f"{prec:.4f}"),
    ("Recall (Positive = crack)",                 f"{rec:.4f}"),
    ("F1 score (Positive = crack)",               f"{f1:.4f}"),
    ("Parameters",                                f"{params/1e6:.2f} M"),
]
summary = pd.DataFrame(summary_rows, columns=["Metric", MODEL_NAME])

# machine-readable row (one line per model; same metric order as above)
results = {
    "model":                 MODEL_NAME,
    "accuracy":              round(acc, 4),
    "latency_gpu_ms_median": round(lat_gpu["median_ms"], 3) if lat_gpu else None,
    "latency_gpu_ms_p95":    round(lat_gpu["p95_ms"], 3) if lat_gpu else None,
    "latency_cpu_ms_median": round(lat_cpu["median_ms"], 3),
    "latency_cpu_ms_p95":    round(lat_cpu["p95_ms"], 3),
    "size_mb":               round(size_mb, 2),
    "gmacs":                 round(gmacs, 3),
    "gflops_2xmacs":         round(2 * gmacs, 3),
    "peak_gpu_mem_mb":       round(peak_mb, 2) if peak_mb is not None else None,
    "precision":             round(prec, 4),
    "recall":                round(rec, 4),
    "f1":                    round(f1, 4),
    "params_M":              round(params / 1e6, 3),
}
# raw confusion-matrix counts (class order: Negative=0, Positive=1), so the
# final multi-model report can be rebuilt from the CSVs alone
tn, fp, fn, tp = (int(v) for v in cm.ravel())
results.update({"cm_tn": tn, "cm_fp": fp, "cm_fn": fn, "cm_tp": tp})
# out_csv = r"d:\VsCode-projects\imageml-benchmark\results_mobilenetv2.csv"
# pd.DataFrame([results]).to_csv(out_csv, index=False)
# print("Saved ->", out_csv)

# why the requested "mAP / Top-5 Accuracy" metrics are absent from this table
print()
print("Note on the requested 'Accuracy (mAP / Top-5 Accuracy)' metric:")
print(" - Top-5 Accuracy is not reported: with only two classes (crack / no crack),")
print("   the correct class is always among the top-5 predictions, so the metric is")
print("   trivially 100% and carries no information. Top-1 accuracy is reported instead.")
print(" - mAP (mean Average Precision) is not reported: it is an object-detection")
print("   metric computed over bounding boxes, and this dataset provides whole-image")
print("   labels only (no bounding-box annotations).")

# presentation: metric names left-aligned, values right-aligned, index hidden
(summary.style
    .hide(axis="index")
    .set_properties(subset=["Metric"], **{"text-align": "left"})
    .set_properties(subset=[MODEL_NAME], **{"text-align": "right"})
    .set_table_styles([
        {"selector": "th.col_heading", "props": "text-align: left;"},
        {"selector": "th.col_heading.col1", "props": "text-align: right;"},
        {"selector": "td, th", "props": "padding: 4px 12px;"},
    ]))

Saved -> d:\VsCode-projects\imageml-benchmark\results_mobilenetv2.csv

Note on the requested 'Accuracy (mAP / Top-5 Accuracy)' metric:
 - Top-5 Accuracy is not reported: with only two classes (crack / no crack),
   the correct class is always among the top-5 predictions, so the metric is
   trivially 100% and carries no information. Top-1 accuracy is reported instead.
 - mAP (mean Average Precision) is not reported: it is an object-detection
   metric computed over bounding boxes, and this dataset provides whole-image
   labels only (no bounding-box annotations).


Metric,MobileNetV2
Accuracy (top-1),99.90 %
"Inference Latency - GPU (batch=1, median)",5.34 ms
"Inference Latency - GPU (batch=1, p95)",5.78 ms
"Inference Latency - CPU (batch=1, median)",17.36 ms
"Inference Latency - CPU (batch=1, p95)",22.53 ms
Model Size (weights on disk),9.15 MB
Computational Cost (GMACs / GFLOPs),0.326 GMACs (~0.652 GFLOPs)
"Memory Usage (peak GPU, single image)",47.3 MB
Precision (Positive = crack),0.9987
Recall (Positive = crack),0.9993
